# 03 · Nodes

**Goal:** by the end you can write a node as a plain Python function that returns a **partial update**, test it by
calling it directly, and register it with a LangGraph builder using `add_node`.

## The idea in plain English

A **node** is one step of work. In LangGraph a node is just a Python function. It receives the whole state (a dict)
and returns a dict containing **only the keys it wants to change**. That returned dict is called a **partial
update**. LangGraph then merges the update into the state for you, the same way `state.update(...)` did in lesson 01.

Because a node is a plain function, you can test it on its own, with no graph at all. Once it works, you register it
with a **builder**, the LangGraph object where you collect the nodes (this lesson) and the edges (lesson 04).

**Analogy:** a clerk doesn't rewrite the whole form. They write their answer on a sticky note ("result: 7") and hand
it back. The office manager (LangGraph) copies the note onto the form.

## Picture

The node reads the state, returns a small update, and LangGraph merges the update into the state.

```mermaid
flowchart LR
    S1["state<br/>a: 3.0, b: 4.0"] --> N["add_node_fn"]
    N --> U["update<br/>result: 7.0"]
    U --> S2["new state<br/>a: 3.0, b: 4.0, result: 7.0"]
    S1 -. "merged with" .-> S2
```

## Step 1: The state schema

The same `CalcState` as lesson 02. Each notebook defines what it needs, so it runs on its own.

In [1]:
from typing import TypedDict


class CalcState(TypedDict):
    a: float
    b: float
    result: float

## Step 2: Write the node

`add_node_fn` takes the state and returns `{"result": ...}`. Notice what it does **not** return: `a` and `b`. It
didn't change them, so it leaves them out. The `-> dict` is a type hint saying "this returns a dict".

In [2]:
def add_node_fn(state: CalcState) -> dict:
    """Add a and b. Return only the key this node changes."""
    return {"result": state["a"] + state["b"]}

## Step 3: Call it directly. It's just Python.

No LangGraph needed to test a node: pass in a dict and look at what comes back. We also print the input afterwards
to show the node didn't change it. A good node *reads* the state and *returns* an update. It doesn't edit the state
it was given.

In [3]:
state: CalcState = {"a": 3.0, "b": 4.0}

update = add_node_fn(state)
print("node returned:", update)
print("input after:  ", state)

node returned: {'result': 7.0}
input after:   {'a': 3.0, 'b': 4.0}


## Step 4: What LangGraph will do with the update

When the graph runs (from lesson 04), LangGraph merges each node's update into the state. For a plain key like
`result`, merging means "the new value replaces the old one". The `{**state, **update}` syntax below builds a new
dict from all the keys of `state`, then all the keys of `update`. Where both have the same key, `update` wins.

This cell is only to show the idea. You won't write this merge yourself. LangGraph does it.

In [4]:
merged = {**state, **update}
print("merged state:", merged)

# The same rule when a key already has a value: the update wins.
print("overwrite:   ", {**{"a": 3.0, "b": 4.0, "result": 0.0}, **update})

merged state: {'a': 3.0, 'b': 4.0, 'result': 7.0}
overwrite:    {'a': 3.0, 'b': 4.0, 'result': 7.0}


## Why return only the keys you change?

- **It says what the node is responsible for.** Reading `return {"result": ...}` tells you this node writes `result`
  and nothing else.
- **It avoids overwriting other nodes' work.** If a node returned the whole state, it could put back an old value
  of a key that another node had just changed. This matters once nodes run in parallel (lesson 08).
- **It's what reducers expect.** In lesson 06 some keys *add* updates to a list instead of replacing them. Returning
  the whole state would add everything twice.

## Step 5: Create a builder and register the node

`StateGraph(CalcState)` creates a **builder**: an empty graph that knows the state schema. `add_node(name, function)`
registers a node under a name. The name is how edges will refer to the node in lesson 04. `builder.nodes` holds the
nodes registered so far.

In [5]:
from langgraph.graph import StateGraph

builder = StateGraph(CalcState)
builder.add_node("add", add_node_fn)

print("nodes in the builder:", list(builder.nodes))

nodes in the builder: ['add']


## Step 6: A shortcut, letting the function name be the node name

If you pass only a function, LangGraph uses the function's name as the node name. Below, a `subtract` function
becomes a node called `"subtract"`. Both forms do the same thing. This tutorial usually gives names explicitly so
they're easy to see.

In [6]:
def subtract(state: CalcState) -> dict:
    return {"result": state["a"] - state["b"]}


builder.add_node(subtract)  # name taken from the function: "subtract"
print("nodes in the builder:", list(builder.nodes))
print("subtract on its own: ", subtract({"a": 10.0, "b": 4.0}))

nodes in the builder: ['add', 'subtract']
subtract on its own:  {'result': 6.0}


## Step 7 (optional): A retry policy

Some steps fail now and then for reasons outside your code, such as a network hiccup. A **retry policy** tells
LangGraph to run a failing node again. `RetryPolicy(max_attempts=3)` means "try up to 3 times in total, waiting a
little longer between each try". Our adding node never fails, so this is just to show where the setting goes.

In [7]:
from langgraph.types import RetryPolicy

builder.add_node("add_with_retry", add_node_fn, retry_policy=RetryPolicy(max_attempts=3))
print("nodes in the builder:", list(builder.nodes))

nodes in the builder: ['add', 'subtract', 'add_with_retry']


We've registered nodes, but there are no edges yet, so the builder can't run anything. Lesson 04 adds the edges,
turns the builder into a runnable graph and runs it.

## What just happened

- **Step 2** wrote `add_node_fn`, a plain function that returns only `result`.
- **Step 3** called it directly. It returned `{'result': 7.0}` and left the input `{'a': 3.0, 'b': 4.0}` unchanged.
- **Step 4** showed the merge LangGraph will do: the state became `{'a': 3.0, 'b': 4.0, 'result': 7.0}`, and an old
  `result: 0.0` was replaced by `7.0`.
- **Step 5** created a builder with `StateGraph(CalcState)` and registered the node as `"add"`.
- **Step 6** registered `subtract` using its function name, so the builder held `['add', 'subtract']`.
- **Step 7** registered a third node with `RetryPolicy(max_attempts=3)`.

## Common mistakes

1. **Changing the state in place instead of returning an update.** A node that does `state["result"] = 7` and
   returns nothing gives **no error**, but the change is thrown away. When the graph runs, `result` is simply
   missing.
   Fix: always `return {"result": ...}`. (Returning nothing is allowed: it means "no update".)
2. **Returning something that isn't a dict.** `return state["a"] + state["b"]` returns a number. When the graph runs
   you get:
   `InvalidUpdateError: Expected dict, got 7.0`.
   Fix: wrap it, as in `return {"result": state["a"] + state["b"]}`.
3. **Registering two nodes with the same name.** Calling `builder.add_node("add", ...)` twice fails straight away:
   `ValueError: Node `add` already present.`
   Fix: give each node its own name.

## Try it

1. Write `multiply_node_fn(state)` that returns the product of `a` and `b` as `result`. Call it directly with
   `{"a": 6.0, "b": 7.0}`.
2. Register it on `builder` under the name `"multiply"` and print `list(builder.nodes)`. Then try registering a
   second node called `"add"`. What happens?

Answers are in [`solutions/exercises_solutions.ipynb`](../solutions/exercises_solutions.ipynb).

## Key terms

New terms from this lesson, also in [GLOSSARY.md](../GLOSSARY.md):

- **Partial update**: the small dict a node returns, holding only the keys it changes.
- **`StateGraph`**: the LangGraph class you create a graph with. `StateGraph(CalcState)` gives you a builder.
- **Builder**: a graph that is still being put together. You add nodes and edges to it, then compile it (lesson 04).
- **`add_node`**: registers a function as a node, under a name.
- **`RetryPolicy`**: a setting that tells LangGraph to re-run a node when it raises an error.